In [ ]:
# ============================================================
# COMPLETE ABLATION STUDY
# SIMPLE EXTRACTIVE QA - NO BiLSTM
# ============================================================
#
# Experiments:
#
# 1. Full Model
#    AmhBERT + Normalization
#
# 2. Without Amharic Pretraining
#    mBERT + Normalization
#
# 3. Without Normalization
#    AmhBERT + Raw Text
#
# All experiments:
#    - Same dataset split
#    - Same QA architecture
#    - Same hyperparameters
#    - Five random seeds
#
# Seeds:
#    13, 21, 42, 77, 123
#
# Metrics:
#    Exact Match (EM)
#    Token-level F1
#
# Statistical analysis:
#    Mean
#    Standard deviation
#    95% confidence interval
#    Paired t-test
#
# ============================================================


# ============================================================
# 1. INSTALL PACKAGES
# ============================================================

!pip install -q transformers datasets accelerate sentencepiece \
    scikit-learn scipy pandas numpy


# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import os
import re
import random
import unicodedata
import numpy as np
import pandas as pd
import torch

from datasets import Dataset

from sklearn.model_selection import train_test_split

from scipy.stats import (
    ttest_rel,
    sem,
    t
)

from transformers import (
    AutoTokenizer,
    AutoModel,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed
)

import torch.nn as nn


# ============================================================
# 3. CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# DATASET
# ------------------------------------------------------------

DATA_PATH = "/content/AmMedQuAD.csv"


# ------------------------------------------------------------
# DATASET COLUMN NAMES
# ------------------------------------------------------------

QUESTION_COL = "question"

CONTEXT_COL = "context"

ANSWER_COL = "answer"


# ------------------------------------------------------------
# MODELS
# ------------------------------------------------------------

# Amharic pretrained BERT
AMHARIC_MODEL = (
    "rasyosef/bert-medium-amharic"
)

# Multilingual BERT
MBERT_MODEL = (
    "bert-base-multilingual-cased"
)


# ------------------------------------------------------------
# RANDOM SEEDS
# ------------------------------------------------------------

SEEDS = [
    13,
    21,
    42,
    77,
    123
]


# ------------------------------------------------------------
# MAX SEQUENCE LENGTH
# ------------------------------------------------------------

MAX_LENGTH = 384


# ------------------------------------------------------------
# TRAINING HYPERPARAMETERS
# ------------------------------------------------------------

LEARNING_RATE = 3e-5

TRAIN_BATCH_SIZE = 8

EVAL_BATCH_SIZE = 8

NUM_EPOCHS = 5

WEIGHT_DECAY = 0.01

WARMUP_RATIO = 0.1

EARLY_STOPPING_PATIENCE = 2


# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

OUTPUT_DIR = (
    "/content/amharic_qa_ablation"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# 4. DEVICE
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "GPU Memory:",
        round(
            torch.cuda.get_device_properties(0)
            .total_memory / 1024**3,
            2
        ),
        "GB"
    )


# ============================================================
# 5. REPRODUCIBILITY
# ============================================================

def seed_everything(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed(seed)

        torch.cuda.manual_seed_all(seed)

    os.environ[
        "PYTHONHASHSEED"
    ] = str(seed)

    set_seed(seed)


# ============================================================
# 6. LOAD DATASET
# ============================================================

df = pd.read_csv(
    DATA_PATH
)

print(
    "Original dataset:",
    df.shape
)

print(
    "\nColumns:"
)

print(
    df.columns.tolist()
)

display(
    df.head()
)


# ============================================================
# 7. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    QUESTION_COL,
    CONTEXT_COL,
    ANSWER_COL
]

for column in required_columns:

    if column not in df.columns:

        raise ValueError(
            f"\nColumn '{column}' "
            f"was not found.\n\n"
            f"Available columns:\n"
            f"{df.columns.tolist()}"
        )


# ============================================================
# 8. SELECT COLUMNS
# ============================================================

df = df[
    [
        QUESTION_COL,
        CONTEXT_COL,
        ANSWER_COL
    ]
].copy()


# ============================================================
# 9. REMOVE MISSING VALUES
# ============================================================

df = df.dropna(
    subset=[
        QUESTION_COL,
        CONTEXT_COL,
        ANSWER_COL
    ]
)


# ============================================================
# 10. CONVERT TO STRING
# ============================================================

df[QUESTION_COL] = (
    df[QUESTION_COL]
    .astype(str)
)

df[CONTEXT_COL] = (
    df[CONTEXT_COL]
    .astype(str)
)

df[ANSWER_COL] = (
    df[ANSWER_COL]
    .astype(str)
)


# ============================================================
# 11. REMOVE EXACT DUPLICATES
# ============================================================

before = len(df)

df = df.drop_duplicates(
    subset=[
        QUESTION_COL,
        CONTEXT_COL,
        ANSWER_COL
    ]
).reset_index(
    drop=True
)

after = len(df)

print(
    f"Removed {before - after} "
    f"exact duplicate examples."
)

print(
    "Final dataset:",
    df.shape
)


# ============================================================
# 12. AMHARIC TEXT NORMALIZATION
# ============================================================

def normalize_amharic(text):

    if text is None:

        return ""

    text = str(text)

    # Unicode normalization
    text = unicodedata.normalize(
        "NFC",
        text
    )

    # Remove zero-width characters
    text = text.replace(
        "\u200b",
        ""
    )

    text = text.replace(
        "\u200c",
        ""
    )

    text = text.replace(
        "\u200d",
        ""
    )

    # Normalize whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    # Remove spaces before punctuation
    text = re.sub(
        r"\s+([።፣፤፥፦፧!?.,;:])",
        r"\1",
        text
    )

    return text


# ============================================================
# 13. NORMALIZE DATASET
# ============================================================

def normalize_dataframe(dataframe):

    data = dataframe.copy()

    data[QUESTION_COL] = (
        data[QUESTION_COL]
        .apply(normalize_amharic)
    )

    data[CONTEXT_COL] = (
        data[CONTEXT_COL]
        .apply(normalize_amharic)
    )

    data[ANSWER_COL] = (
        data[ANSWER_COL]
        .apply(normalize_amharic)
    )

    return data


# ============================================================
# 14. CREATE FIXED DATA SPLIT
# ============================================================
#
# IMPORTANT:
# The same split is used for EVERY ablation.
#
# This is necessary so that differences in performance
# are attributable to the ablated component rather than
# different train/test partitions.
#
# ============================================================

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42
)


# Reset indexes

train_df = train_df.reset_index(
    drop=True
)

valid_df = valid_df.reset_index(
    drop=True
)

test_df = test_df.reset_index(
    drop=True
)


print(
    "\nDataset split:"
)

print(
    "Training:",
    len(train_df)
)

print(
    "Validation:",
    len(valid_df)
)

print(
    "Test:",
    len(test_df)
)


# ============================================================
# 15. NORMALIZED DATA
# ============================================================

train_norm = normalize_dataframe(
    train_df
)

valid_norm = normalize_dataframe(
    valid_df
)

test_norm = normalize_dataframe(
    test_df
)


# ============================================================
# 16. FIND ANSWER START POSITION
# ============================================================

def find_answer_start(
    context,
    answer
):

    # Exact answer
    start = context.find(
        answer
    )

    if start != -1:

        return start


    # Normalized matching
    norm_context = normalize_amharic(
        context
    )

    norm_answer = normalize_amharic(
        answer
    )

    start = norm_context.find(
        norm_answer
    )

    if start != -1:

        return start


    return -1


# ============================================================
# 17. EXTRACTIVE QA MODEL
# ============================================================
#
# IMPORTANT:
# NO BiLSTM.
#
# Architecture:
#
# Encoder
#    ↓
# Dropout
#    ↓
# Linear QA head
#    ↓
# Start / End logits
#
# ============================================================

class SimpleExtractiveQA(
    nn.Module
):

    def __init__(
        self,
        model_name,
        dropout=0.1
    ):

        super().__init__()


        # Transformer encoder
        self.encoder = (
            AutoModel.from_pretrained(
                model_name
            )
        )


        hidden_size = (
            self.encoder
            .config
            .hidden_size
        )


        # Dropout
        self.dropout = nn.Dropout(
            dropout
        )


        # QA output layer
        #
        # Output:
        # 0 = start position
        # 1 = end position

        self.qa_outputs = nn.Linear(
            hidden_size,
            2
        )


    def forward(
        self,
        input_ids,
        attention_mask=None,
        start_positions=None,
        end_positions=None
    ):


        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )


        sequence_output = (
            outputs.last_hidden_state
        )


        sequence_output = (
            self.dropout(
                sequence_output
            )
        )


        logits = self.qa_outputs(
            sequence_output
        )


        start_logits = (
            logits[:, :, 0]
        )

        end_logits = (
            logits[:, :, 1]
        )


        loss = None


        if (
            start_positions is not None
            and end_positions is not None
        ):


            loss_function = (
                nn.CrossEntropyLoss()
            )


            start_loss = (
                loss_function(
                    start_logits,
                    start_positions
                )
            )


            end_loss = (
                loss_function(
                    end_logits,
                    end_positions
                )
            )


            loss = (
                start_loss
                + end_loss
            ) / 2


        return {
            "loss": loss,
            "start_logits": start_logits,
            "end_logits": end_logits
        }


# ============================================================
# 18. TOKENIZE DATASET
# ============================================================

def prepare_features(
    examples,
    tokenizer
):

    questions = [
        q.strip()
        for q in examples[
            QUESTION_COL
        ]
    ]

    contexts = examples[
        CONTEXT_COL
    ]

    answers = examples[
        ANSWER_COL
    ]


    tokenized = tokenizer(
        questions,
        contexts,
        max_length=MAX_LENGTH,
        truncation="only_second",
        padding="max_length",
        return_offsets_mapping=True
    )


    start_positions = []

    end_positions = []


    for i in range(
        len(questions)
    ):


        context = contexts[i]

        answer = answers[i]


        answer_start = (
            find_answer_start(
                context,
                answer
            )
        )


        # Answer not found
        if answer_start == -1:

            start_positions.append(0)

            end_positions.append(0)

            continue


        answer_end = (
            answer_start
            + len(answer)
        )


        offsets = (
            tokenized[
                "offset_mapping"
            ][i]
        )


        sequence_ids = (
            tokenized.sequence_ids(i)
        )


        # Find context token boundaries

        context_start = None

        context_end = None


        for idx, seq_id in enumerate(
            sequence_ids
        ):

            if seq_id == 1:

                if context_start is None:

                    context_start = idx

                context_end = idx


        if context_start is None:

            start_positions.append(0)

            end_positions.append(0)

            continue


        # ----------------------------------------------------
        # Find start token
        # ----------------------------------------------------

        token_start = (
            context_start
        )


        while (
            token_start <= context_end
            and
            offsets[token_start][0]
            <= answer_start
        ):

            token_start += 1


        token_start -= 1


        # ----------------------------------------------------
        # Find end token
        # ----------------------------------------------------

        token_end = (
            context_end
        )


        while (
            token_end >= context_start
            and
            offsets[token_end][1]
            >= answer_end
        ):

            token_end -= 1


        token_end += 1


        # ----------------------------------------------------
        # Validate alignment
        # ----------------------------------------------------

        if (
            token_start < context_start
            or
            token_end > context_end
        ):

            start_positions.append(0)

            end_positions.append(0)

        else:

            start_positions.append(
                token_start
            )

            end_positions.append(
                token_end
            )


    tokenized[
        "start_positions"
    ] = start_positions


    tokenized[
        "end_positions"
    ] = end_positions


    # Offset mapping is not needed during training

    tokenized.pop(
        "offset_mapping"
    )


    return tokenized


# ============================================================
# 19. CUSTOM TRAINER
# ============================================================

class ExtractiveQATrainer(
    Trainer
):


    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):


        outputs = model(
            input_ids=inputs[
                "input_ids"
            ],

            attention_mask=inputs[
                "attention_mask"
            ],

            start_positions=inputs[
                "start_positions"
            ],

            end_positions=inputs[
                "end_positions"
            ]
        )


        loss = outputs[
            "loss"
        ]


        if return_outputs:

            return (
                loss,
                outputs
            )


        return loss


# ============================================================
# 20. NORMALIZE ANSWER FOR METRICS
# ============================================================

def normalize_metric_text(
    text
):

    text = str(text)

    text = normalize_amharic(
        text
    )

    return text.lower().strip()


# ============================================================
# 21. EXACT MATCH
# ============================================================

def compute_exact_match(
    prediction,
    reference
):

    prediction = (
        normalize_metric_text(
            prediction
        )
    )

    reference = (
        normalize_metric_text(
            reference
        )
    )

    return int(
        prediction == reference
    )


# ============================================================
# 22. TOKEN F1
# ============================================================

def compute_f1(
    prediction,
    reference
):

    prediction_tokens = (
        normalize_metric_text(
            prediction
        ).split()
    )

    reference_tokens = (
        normalize_metric_text(
            reference
        ).split()
    )


    if (
        len(prediction_tokens) == 0
        or
        len(reference_tokens) == 0
    ):

        return float(
            prediction_tokens
            == reference_tokens
        )


    common = {}

    for token in prediction_tokens:

        if token in reference_tokens:

            common[token] = (
                common.get(
                    token,
                    0
                )
                + 1
            )


    num_same = sum(
        min(
            count,
            reference_tokens.count(token)
        )
        for token, count
        in common.items()
    )


    if num_same == 0:

        return 0.0


    precision = (
        num_same
        /
        len(prediction_tokens)
    )


    recall = (
        num_same
        /
        len(reference_tokens)
    )


    return (
        2
        * precision
        * recall
        /
        (precision + recall)
    )


# ============================================================
# 23. GENERATE ANSWER PREDICTIONS
# ============================================================

def predict_answers(
    model,
    tokenizer,
    dataframe
):


    model.eval()

    predictions = []

    references = []


    for _, row in (
        dataframe
        .reset_index(drop=True)
        .iterrows()
    ):


        question = row[
            QUESTION_COL
        ]

        context = row[
            CONTEXT_COL
        ]

        reference = row[
            ANSWER_COL
        ]


        encoded = tokenizer(
            question,
            context,
            max_length=MAX_LENGTH,
            truncation="only_second",
            padding="max_length",
            return_offsets_mapping=True,
            return_tensors="pt"
        )


        offsets = (
            encoded[
                "offset_mapping"
            ][0]
            .cpu()
            .numpy()
        )


        input_ids = (
            encoded[
                "input_ids"
            ].to(device)
        )


        attention_mask = (
            encoded[
                "attention_mask"
            ].to(device)
        )


        with torch.no_grad():

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )


        start_logits = (
            outputs[
                "start_logits"
            ][0]
            .cpu()
            .numpy()
        )


        end_logits = (
            outputs[
                "end_logits"
            ][0]
            .cpu()
            .numpy()
        )


        # ----------------------------------------------------
        # Select best start/end span
        # ----------------------------------------------------

        best_score = (
            -float("inf")
        )

        best_start = 0

        best_end = 0


        start_indices = (
            np.argsort(
                start_logits
            )[-20:]
        )


        end_indices = (
            np.argsort(
                end_logits
            )[-20:]
        )


        for start in start_indices:

            for end in end_indices:

                if end < start:

                    continue


                # Prevent extremely long spans

                if (
                    end - start
                    > 30
                ):

                    continue


                score = (
                    start_logits[start]
                    +
                    end_logits[end]
                )


                if score > best_score:

                    best_score = score

                    best_start = start

                    best_end = end


        start_char = offsets[
            best_start
        ][0]


        end_char = offsets[
            best_end
        ][1]


        prediction = context[
            start_char:end_char
        ]


        predictions.append(
            prediction
        )


        references.append(
            reference
        )


    return (
        predictions,
        references
    )


# ============================================================
# 24. EVALUATE MODEL
# ============================================================

def evaluate_model(
    model,
    tokenizer,
    dataframe
):


    predictions, references = (
        predict_answers(
            model,
            tokenizer,
            dataframe
        )
    )


    em_scores = []

    f1_scores = []


    for prediction, reference in zip(
        predictions,
        references
    ):


        em = compute_exact_match(
            prediction,
            reference
        )


        f1 = compute_f1(
            prediction,
            reference
        )


        em_scores.append(
            em
        )


        f1_scores.append(
            f1
        )


    return {

        "EM":
            np.mean(em_scores) * 100,

        "F1":
            np.mean(f1_scores) * 100
    }


# ============================================================
# 25. RUN ONE EXPERIMENT
# ============================================================

def run_experiment(
    experiment_name,
    model_name,
    use_normalization,
    seed
):


    print("\n")
    print("=" * 70)

    print(
        "Experiment:",
        experiment_name
    )

    print(
        "Model:",
        model_name
    )

    print(
        "Normalization:",
        use_normalization
    )

    print(
        "Seed:",
        seed
    )

    print("=" * 70)


    # Reproducibility

    seed_everything(
        seed
    )


    # --------------------------------------------------------
    # Select dataset version
    # --------------------------------------------------------

    if use_normalization:

        train_data = train_norm

        valid_data = valid_norm

        test_data = test_norm

    else:

        train_data = train_df

        valid_data = valid_df

        test_data = test_df


    # --------------------------------------------------------
    # Tokenizer
    # --------------------------------------------------------

    tokenizer = (
        AutoTokenizer
        .from_pretrained(
            model_name
        )
    )


    # --------------------------------------------------------
    # Convert to HF Dataset
    # --------------------------------------------------------

    train_dataset = (
        Dataset.from_pandas(
            train_data,
            preserve_index=False
        )
    )


    valid_dataset = (
        Dataset.from_pandas(
            valid_data,
            preserve_index=False
        )
    )


    # --------------------------------------------------------
    # Tokenization
    # --------------------------------------------------------

    train_dataset = (
        train_dataset.map(
            lambda examples:
            prepare_features(
                examples,
                tokenizer
            ),
            batched=True,
            remove_columns=[
                QUESTION_COL,
                CONTEXT_COL,
                ANSWER_COL
            ]
        )
    )


    valid_dataset = (
        valid_dataset.map(
            lambda examples:
            prepare_features(
                examples,
                tokenizer
            ),
            batched=True,
            remove_columns=[
                QUESTION_COL,
                CONTEXT_COL,
                ANSWER_COL
            ]
        )
    )


    # --------------------------------------------------------
    # Create simple QA model
    # --------------------------------------------------------

    model = SimpleExtractiveQA(
        model_name=model_name
    )


    model.to(device)


    # --------------------------------------------------------
    # Output directory
    # --------------------------------------------------------

    experiment_dir = os.path.join(
        OUTPUT_DIR,
        experiment_name.replace(
            " ",
            "_"
        ),
        f"seed_{seed}"
    )


    os.makedirs(
        experiment_dir,
        exist_ok=True
    )


    # --------------------------------------------------------
    # Training arguments
    # --------------------------------------------------------

    training_args = (
        TrainingArguments(

            output_dir=
                experiment_dir,

            learning_rate=
                LEARNING_RATE,

            per_device_train_batch_size=
                TRAIN_BATCH_SIZE,

            per_device_eval_batch_size=
                EVAL_BATCH_SIZE,

            num_train_epochs=
                NUM_EPOCHS,

            weight_decay=
                WEIGHT_DECAY,

            warmup_ratio=
                WARMUP_RATIO,

            eval_strategy=
                "epoch",

            save_strategy=
                "epoch",

            logging_strategy=
                "epoch",

            load_best_model_at_end=
                True,

            metric_for_best_model=
                "eval_loss",

            greater_is_better=
                False,

            save_total_limit=
                1,

            report_to=
                "none",

            fp16=
                torch.cuda.is_available(),

            seed=
                seed
        )
    )


    # --------------------------------------------------------
    # Trainer
    # --------------------------------------------------------

    trainer = ExtractiveQATrainer(

        model=model,

        args=training_args,

        train_dataset=
            train_dataset,

        eval_dataset=
            valid_dataset,

        tokenizer=
            tokenizer,

        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=
                    EARLY_STOPPING_PATIENCE
            )
        ]
    )


    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    trainer.train()


    # --------------------------------------------------------
    # Best model
    # --------------------------------------------------------

    model = trainer.model

    model.to(device)


    # --------------------------------------------------------
    # Test evaluation
    # --------------------------------------------------------

    metrics = evaluate_model(
        model,
        tokenizer,
        test_data
    )


    print(
        "\nRESULT:"
    )

    print(
        f"EM = {metrics['EM']:.2f}"
    )

    print(
        f"F1 = {metrics['F1']:.2f}"
    )


    # --------------------------------------------------------
    # Return result
    # --------------------------------------------------------

    return {

        "Experiment":
            experiment_name,

        "Model":
            model_name,

        "Normalization":
            use_normalization,

        "Seed":
            seed,

        "EM":
            metrics["EM"],

        "F1":
            metrics["F1"]
    }


# ============================================================
# 26. DEFINE ABLATION EXPERIMENTS
# ============================================================
#
# ONLY THREE CONDITIONS
#
# NO BiLSTM
#
# ------------------------------------------------------------
#
# Full Model:
# AmhBERT + normalization
#
# Without Amharic pretraining:
# mBERT + normalization
#
# Without normalization:
# AmhBERT + raw text
#
# ============================================================

experiments = {

    "Full Model": {

        "model":
            AMHARIC_MODEL,

        "normalization":
            True
    },


    "Without Amharic Pretraining": {

        "model":
            MBERT_MODEL,

        "normalization":
            True
    },


    "Without Normalization": {

        "model":
            AMHARIC_MODEL,

        "normalization":
            False
    }
}


# ============================================================
# 27. RUN FIVE SEEDS
# ============================================================

all_results = []


for experiment_name, config in (
    experiments.items()
):


    for seed in SEEDS:


        try:


            result = run_experiment(

                experiment_name=
                    experiment_name,

                model_name=
                    config["model"],

                use_normalization=
                    config["normalization"],

                seed=
                    seed
            )


            all_results.append(
                result
            )


            # Save after every run
            results_temp = (
                pd.DataFrame(
                    all_results
                )
            )


            results_temp.to_csv(
                os.path.join(
                    OUTPUT_DIR,
                    "ablation_raw_results.csv"
                ),
                index=False
            )


        except Exception as e:


            print(
                "\nERROR:"
            )

            print(
                experiment_name
            )

            print(
                "Seed:",
                seed
            )

            print(
                str(e)
            )


# ============================================================
# 28. LOAD RAW RESULTS
# ============================================================

results_df = pd.DataFrame(
    all_results
)


print(
    "\nRAW RESULTS"
)

display(
    results_df
)


# ============================================================
# 29. SAVE RAW RESULTS
# ============================================================

results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "ablation_raw_results.csv"
    ),
    index=False
)


# ============================================================
# 30. MEAN AND STANDARD DEVIATION
# ============================================================

summary = (
    results_df
    .groupby("Experiment")
    .agg(

        EM_Mean=(
            "EM",
            "mean"
        ),

        EM_STD=(
            "EM",
            "std"
        ),

        F1_Mean=(
            "F1",
            "mean"
        ),

        F1_STD=(
            "F1",
            "std"
        ),

        N=(
            "EM",
            "count"
        )
    )
    .reset_index()
)


# ============================================================
# 31. 95% CONFIDENCE INTERVAL
# ============================================================

def confidence_interval(
    values,
    confidence=0.95
):

    values = np.array(
        values,
        dtype=float
    )

    n = len(values)

    mean = np.mean(
        values
    )

    if n < 2:

        return (
            mean,
            mean
        )

    standard_error = sem(
        values
    )

    margin = (
        t.ppf(
            (1 + confidence) / 2,
            n - 1
        )
        *
        standard_error
    )

    return (
        mean - margin,
        mean + margin
    )


em_lower = []

em_upper = []

f1_lower = []
f1_upper = []
for experiment in summary[
    "Experiment"].values:
    subset = results_df[
        results_df["Experiment"]== experiment]
    low, high = (confidence_interval(
            subset["EM"]
        )
    )
    em_lower.append(
        low
    )

    em_upper.append(
        high
    )


    low, high = (
        confidence_interval(
            subset["F1"]
        )
    )

    f1_lower.append(
        low
    )

    f1_upper.append(
        high
    )


summary[
    "EM_CI95_Lower"
] = em_lower


summary[
    "EM_CI95_Upper"
] = em_upper


summary[
    "F1_CI95_Lower"
] = f1_lower


summary[
    "F1_CI95_Upper"
] = f1_upper


# ============================================================
# 32. MEAN ± SD FORMAT
# ============================================================

summary[
    "EM_Mean_SD"
] = (

    summary[
        "EM_Mean"
    ].round(2).astype(str)

    + " ± "

    + summary[
        "EM_STD"
    ].round(2).astype(str)
)


summary[
    "F1_Mean_SD"
] = (

    summary[
        "F1_Mean"
    ].round(2).astype(str)

    + " ± "

    + summary[
        "F1_STD"
    ].round(2).astype(str)
)


# ============================================================
# 33. DISPLAY SUMMARY
# ============================================================

print(
    "\nABLATION SUMMARY"
)

display(
    summary[
        [
            "Experiment",
            "N",
            "EM_Mean_SD",
            "F1_Mean_SD",
            "EM_CI95_Lower",
            "EM_CI95_Upper",
            "F1_CI95_Lower",
            "F1_CI95_Upper"
        ]
    ]
)


# ============================================================
# 34. SAVE SUMMARY
# ============================================================

summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "ablation_summary.csv"
    ),
    index=False
)


# ============================================================
# 35. PAIRED SIGNIFICANCE TEST
# ============================================================
#
# Full Model is compared against:
#
# 1. Without Amharic Pretraining
# 2. Without Normalization
#
# The SAME five seeds are paired.
#
# ============================================================

full = results_df[
    results_df[
        "Experiment"
    ]
    == "Full Model"
].sort_values(
    "Seed"
)


significance_results = []


comparisons = [
    "Without Amharic Pretraining",
    "Without Normalization"
]


for comparison_name in comparisons:


    ablation = results_df[
        results_df[
            "Experiment"
        ]
        == comparison_name
    ].sort_values(
        "Seed"
    )


    merged = pd.merge(

        full[
            [
                "Seed",
                "EM",
                "F1"
            ]
        ],

        ablation[
            [
                "Seed",
                "EM",
                "F1"
            ]
        ],

        on="Seed",

        suffixes=(
            "_Full",
            "_Ablation"
        )
    )


    if len(merged) >= 2:


        # EM test
        em_test = ttest_rel(
            merged["EM_Full"],
            merged["EM_Ablation"]
        )


        # F1 test
        f1_test = ttest_rel(
            merged["F1_Full"],
            merged["F1_Ablation"]
        )


        significance_results.append({

            "Comparison":
                (
                    "Full Model vs "
                    + comparison_name
                ),

            "N":
                len(merged),

            "EM_Full_Mean":
                merged[
                    "EM_Full"
                ].mean(),

            "EM_Ablation_Mean":
                merged[
                    "EM_Ablation"
                ].mean(),

            "EM_p_value":
                em_test.pvalue,

            "F1_Full_Mean":
                merged[
                    "F1_Full"
                ].mean(),

            "F1_Ablation_Mean":
                merged[
                    "F1_Ablation"
                ].mean(),

            "F1_p_value":
                f1_test.pvalue
        })


significance_df = pd.DataFrame(
    significance_results
)


# ============================================================
# 36. SIGNIFICANCE SYMBOL
# ============================================================

def significance_symbol(
    p
):

    if p < 0.001:

        return "***"

    elif p < 0.01:

        return "**"

    elif p < 0.05:

        return "*"

    else:

        return "ns"


if len(significance_df) > 0:


    significance_df[
        "EM_Significance"
    ] = (
        significance_df[
            "EM_p_value"
        ]
        .apply(
            significance_symbol
        )
    )


    significance_df[
        "F1_Significance"
    ] = (
        significance_df[
            "F1_p_value"
        ]
        .apply(
            significance_symbol
        )
    )


print(
    "\nSTATISTICAL SIGNIFICANCE"
)

display(
    significance_df
)


# ============================================================
# 37. SAVE SIGNIFICANCE RESULTS
# ============================================================

significance_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "ablation_significance.csv"
    ),
    index=False
)
# ============================================================
# 38. CREATE FINAL PAPER TABLE
# ============================================================
paper_table = summary[["Experiment","EM_Mean_SD","F1_Mean_SD"]].copy()
paper_table.columns = ["Ablation Configuration","EM (%)","F1 (%)"]
print("\nFINAL TABLE FOR MANUSCRIPT")
display( paper_table)
# ============================================================
# 39. SAVE PAPER TABLE
# ============================================================

paper_table.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "ablation_paper_table.csv"
    ),
    index=False
)


# ============================================================
# 40. PRINT RESULTS FILES
# ============================================================

print("\n================================================")
print("ABLATION STUDY COMPLETE")
print( "================================================")
print("\nRaw results:")
print(os.path.join(OUTPUT_DIR,"ablation_raw_results.csv"))
print("\nSummary:")
print(os.path.join(OUTPUT_DIR,"ablation_summary.csv"))
print("\nSignificance:")
print(os.path.join(OUTPUT_DIR,"ablation_significance.csv"))
print("\nPaper table:")
print(os.path.join(OUTPUT_DIR,"ablation_paper_table.csv"))